In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy.stats import pearsonr

class ExperimentalDataAnalyzer:
    def __init__(self, a1, a2, a3, h1, m1, m2, m3):
        """
        Inizializza l'analizzatore con i parametri del modello stimato
        """
        self.a1, self.a2, self.a3 = a1, a2, a3
        self.h1 = h1  # Tipicamente 0
        self.m1, self.m2, self.m3 = m1, m2, m3
        
        # Soglie per valutazione qualitativa (basate su esperienza)
        self.thresholds = {
            'relative_error_excellent': 5.0,     # %
            'relative_error_good': 10.0,         # %
            'relative_error_acceptable': 20.0,   # %
            'correlation_excellent': 0.95,
            'correlation_good': 0.90,
            'correlation_acceptable': 0.80
        }
        
    def load_experimental_data(self, file_path):
        """
        Carica i dati sperimentali dal file di testo
        """
        try:
            # Legge il file saltando le prime 2 righe (header)
            data = pd.read_csv(file_path, sep='\t', skiprows=2)
            
            # Rinomina le colonne per comodità
            data.columns = ['Time', 'Pd_nm', 'Fn_mN', 'FnRef_mN', 'SegmentID', 'Res_ohm']
            
            # Converte gli spostamenti da nm a unità coerenti (manteniamo nm)
            data['Displacement_nm'] = data['Pd_nm']
            
            print(f"Dati caricati: {len(data)} punti")
            print(f"Range temporale: {data['Time'].min():.1f} - {data['Time'].max():.1f} s")
            print(f"Range forza: {data['Fn_mN'].min():.4f} - {data['Fn_mN'].max():.4f} N")
            print(f"Range spostamento: {data['Displacement_nm'].min():.1f} - {data['Displacement_nm'].max():.1f} nm")
            
            return data
            
        except Exception as e:
            print(f"Errore nel caricamento dei dati: {e}")
            return None
    
    def calculate_theoretical_displacement(self, force_array, phase_array):
        """
        Calcola h seguendo ESATTAMENTE il modello (3.4) con continuità corretta
        """
        h_array = np.zeros_like(force_array)
        
        # Valori per la continuità (h_k dall'ultimo punto del segmento precedente)
        h_k = 0  # h_k per unloading
        h2 = 0   # h2 calcolato per continuità in unloading
        h3 = 0   # h3 calcolato per continuità in unload_light
        
        # Costanti c1 e c2 (valori alla fine delle fasi di loading)
        c1 = 0
        c2 = 0
        
        for i, (F, segment) in enumerate(zip(force_array, phase_array)):
            
            if segment == 1 or segment == 0:  # Loading: μᵢ = a₁·(hᵢ - h₁)^m₁:
                h_array[i] = (F / self.a1)**(1/self.m1) + self.h1 
                    
                # Memorizza c1 (ultimo valore di loading)
                if i == len(force_array)-1 or phase_array[i+1] != 1:
                    c1 = F
                    h_k = h_array[i]  # h_k è la profondità alla fine del loading
                
            elif segment == 2:  # Hold1: μᵢ = c₁ (costante)
                h_array[i] = h_k
                
            elif segment == 3:  # Unloading
                # Prima volta in unloading: calcola h2 per continuità
                if i > 0 and phase_array[i-1] == 2:
                    # h2 = hᵢ - (a₁·(h_k-h2)^m₁/a₂)^(1/m₂)
                    # dove h_k è la profondità dell'ultimo punto del segmento precedente
                    term = self.a1 * (h_k - self.h1)**self.m1
                    h2 = h_array[i-1] - (term / self.a2)**(1/self.m2)
                
                h_array[i] = (F / self.a2)**(1/self.m2) + h2
                    
                # Memorizza c2 (ultimo valore di unloading)  
                if i == len(force_array)-1 or phase_array[i+1] != 3:
                    c2 = F
                    h_k = h_array[i]  # Aggiorna h_k per la prossima fase
                
            elif segment == 4:  # Hold2: μᵢ = c₂ (costante)
                h_array[i] = h_k
                
            elif segment == 5 or segment == 6:  # Unload_light
                # Prima volta in unload_light: calcola h3 per continuità
                if i > 0 and phase_array[i-1] == 4:
                    # h3 = hᵢ - (a₂·(h_k-h2)^m₂/a₃)^(1/m₃)
                    # dove h_k è la forza dell'ultimo punto del segmento precedente
                    term = self.a2 * (h_k - h2)**self.m2
                    h3 = h_array[i-1] - (term / self.a3)**(1/self.m3)

                h_array[i] = (F / self.a3)**(1/self.m3) + h3
                    
            # Non permettere spostamenti negativi
            h_array[i] = max(0, h_array[i])
            
        return h_array
    
    def calculate_advanced_metrics(self, experimental, theoretical, phases):
        """
        Calcola metriche avanzate focalizzate su forma della curva e consistenza
        """
        metrics = {}
        
        # 1. Errore relativo medio globale
        valid_mask = experimental > 0.1  # Evita divisioni per valori troppo piccoli
        if np.any(valid_mask):
            relative_error_global = np.mean(np.abs((experimental[valid_mask] - theoretical[valid_mask]) / experimental[valid_mask])) * 100
        else:
            relative_error_global = float('inf')
        
        # 2. Correlazione della forma (derivate)
        if len(experimental) > 1:
            exp_diff = np.diff(experimental)
            theo_diff = np.diff(theoretical)
            if np.std(exp_diff) > 0 and np.std(theo_diff) > 0:
                shape_correlation, _ = pearsonr(exp_diff, theo_diff)
            else:
                shape_correlation = 0
        else:
            shape_correlation = 0
        
        metrics['global'] = {
            'RelativeError_percent': relative_error_global,
            'ShapeCorrelation': shape_correlation
        }
        
        # 3. Metriche per fase
        phase_names = {1: 'Loading', 2: 'Hold1', 3: 'Unloading', 4: 'Hold2', 5: 'Unload_light'}
        
        for phase in sorted(np.unique(phases)):
            if phase in phase_names:
                mask = phases == phase
                if np.any(mask) and np.sum(mask) > 2:  # Almeno 3 punti per calcolare metriche
                    exp_phase = experimental[mask]
                    theo_phase = theoretical[mask]
                    
                    # Errore relativo medio per questa fase
                    valid_phase_mask = exp_phase > 0.1
                    if np.any(valid_phase_mask):
                        rel_error_phase = np.mean(np.abs((exp_phase[valid_phase_mask] - theo_phase[valid_phase_mask]) / exp_phase[valid_phase_mask])) * 100
                    else:
                        rel_error_phase = float('inf')
                    
                    metrics[f'phase_{phase}_{phase_names[phase]}'] = {
                        'RelativeError_percent': rel_error_phase,
                        'NumPoints': np.sum(mask)
                    }
        
        return metrics
    
    
    def create_plots(self, data, save_path, file_name_base):
        """
        Crea e salva i grafici di confronto, escludendo le fasi di preparazione (0) e la fase 6.
        """
        # Filtra i dati per escludere le fasi 0 e 6 solo per la visualizzazione
        plot_data = data[~data['SegmentID'].isin([0, 6])].copy()
        
        # Definisce colori e nomi per le fasi rilevanti
        phase_colors = {1: 'red', 2: 'orange', 3: 'green', 4: 'blue', 5: 'purple'}
        phase_names = {1: 'Loading', 2: 'Hold1', 3: 'Unloading', 4: 'Hold2', 5: 'Unload_light'}
        
        # === Plot 1: Curva Forza-Spostamento (F-h) ===
        fig1, ax1 = plt.subplots(figsize=(10, 8))
        
        # Itera sulle fasi per colorare la curva F-h
        for phase in sorted(plot_data['SegmentID'].unique()):
            mask = plot_data['SegmentID'] == phase
            # Curva sperimentale
            ax1.plot(plot_data.loc[mask, 'Fn_mN'], plot_data.loc[mask, 'Displacement_nm'],
                     color=phase_colors.get(phase, 'black'), linewidth=3, alpha=0.7,
                     label=f'{phase_names.get(phase, f"Fase {phase}")} - Sper.')
            # Curva teorica
            ax1.plot(plot_data.loc[mask, 'Fn_mN'], plot_data.loc[mask, 'Theoretical_nm'],
                     color=phase_colors.get(phase, 'black'), linewidth=2, linestyle='--', alpha=0.9)
        
        ax1.set_xlabel('Forza F (N)')
        ax1.set_ylabel('Spostamento h (nm)')
        ax1.set_title('Curva F-h: Sperimentale (linea) vs Teorico (tratteggio)')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plot1_path = os.path.join(save_path, f"{file_name_base}_curva_F-h.jpg")
        plt.savefig(plot1_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        # === Plot 2: Spostamento nel Tempo ===
        fig2, ax2 = plt.subplots(figsize=(12, 6))
        
        ax2.plot(plot_data['Time'], plot_data['Displacement_nm'], 'b-', linewidth=3, alpha=0.7, label='Sperimentale')
        ax2.plot(plot_data['Time'], plot_data['Theoretical_nm'], 'r--', linewidth=2, label='Teorico')
        ax2.set_xlabel('Tempo (s)')
        ax2.set_ylabel('Spostamento h (nm)')
        ax2.set_title('Confronto Spostamenti nel Tempo: Sperimentale vs Teorico')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plot2_path = os.path.join(save_path, f"{file_name_base}_spostamento_tempo.jpg")
        plt.savefig(plot2_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        return plot1_path, plot2_path

    def save_report(self, metrics, data, save_path, file_name_base):
        """
        Salva un report completo in formato testo
        """
        report_path = os.path.join(save_path, f"{file_name_base}_report.txt")
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("REPORT ANALISI CONFRONTO SPERIMENTALE vs MODELLO TEORICO\n")
            f.write("="*80 + "\n\n")
            
            # Informazioni generali
            f.write("INFORMAZIONI GENERALI:\n")
            f.write(f"    Numero di punti analizzati: {len(data)}\n")
            f.write(f"    Range temporale: {data['Time'].min():.1f} - {data['Time'].max():.1f} s\n")
            f.write(f"    Range forza: {data['Fn_mN'].min():.4f} - {data['Fn_mN'].max():.4f} N\n")
            f.write(f"    Range spostamento: {data['Displacement_nm'].min():.1f} - {data['Displacement_nm'].max():.1f} nm\n\n")
            
            # Metriche dettagliate
            f.write("METRICHE GLOBALI:\n")
            f.write(f"    Errore relativo medio: {metrics['global']['RelativeError_percent']:.2f}%\n")
            f.write(f"    Correlazione forma: {metrics['global']['ShapeCorrelation']:.4f}\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("INTERPRETAZIONE RISULTATI:\n")
            f.write("="*80 + "\n")
            f.write("Errore relativo < 5%: Precisione eccellente\n")
            f.write("Errore relativo < 10%: Precisione buona\n")
            f.write("Errore relativo < 20%: Precisione accettabile\n")
            f.write("Errore relativo > 20%: Verificare setup sperimentale\n\n")
            f.write("Correlazione forma > 0.95: Forma eccellente\n")
            f.write("Correlazione forma > 0.90: Forma buona\n")
            f.write("Correlazione forma > 0.80: Forma accettabile\n")
            f.write("Correlazione forma < 0.80: Verificare condizioni sperimentali\n")
        
        return report_path
    
    def run_complete_analysis(self):
        """
        Esegue l'analisi completa
        """
        print("ANALIZZATORE DATI SPERIMENTALI - CONFRONTO CON MODELLO TEORICO")
        print("=" * 70)
        
        # Input file dati
        data_file = input("Inserisci il percorso completo del file dati sperimentali:\n> ").strip('"')
        
        # Input cartella di salvataggio
        save_folder = input("Inserisci il percorso della cartella dove salvare i risultati:\n> ").strip('"')
        os.makedirs(save_folder, exist_ok=True)
        
        # Nome base per i file
        file_base = "analisi_" + os.path.splitext(os.path.basename(data_file))[0]
        
        print(f"\nFile dati: {data_file}")
        print(f"Cartella output: {save_folder}\n")
        
        # Carica dati
        data = self.load_experimental_data(data_file)
        if data is None:
            return
        
        # Calcola modello teorico
        print("\nCalcolo modello teorico...")
        h_theoretical = self.calculate_theoretical_displacement(
            data['Fn_mN'].values, data['SegmentID'].values)
        data['Theoretical_nm'] = h_theoretical
        
        # Calcola metriche
        print("Calcolo metriche...")
        metrics = self.calculate_advanced_metrics(
            data['Displacement_nm'].values, data['Theoretical_nm'].values, data['SegmentID'].values)
        
        # Crea grafici
        print("Generazione grafici...")
        plot1_path, plot2_path = self.create_plots(data, save_folder, file_base)
        
        # Salva report
        print("Salvataggio report...")
        report_path = self.save_report(metrics, data, save_folder, file_base)
        
        # Risultati finali
        print("\n" + "="*70)
        print("ANALISI COMPLETATA")
        print("="*70)

        print(f"\nMetriche principali:")
        print(f"    Errore relativo medio: {metrics['global']['RelativeError_percent']:.2f}%")
        print(f"    Correlazione forma: {metrics['global']['ShapeCorrelation']:.4f}")
        
        print(f"\nFile generati:")
        print(f"    Confronto temporale: {plot1_path}")
        print(f"    Analisi errore: {plot2_path}")
        print(f"    Report completo: {report_path}")
        
        return data, metrics

# Esempio di utilizzo
if __name__ == "__main__":
    # Chiede all'utente se i dati sono sotto controllo termico
    thermal_control_input = ''
    while thermal_control_input.lower() not in ['s', 'n']:
        thermal_control_input = input("I dati sono stati ottenuti sotto controllo termico? (s/n): ")

    if thermal_control_input.lower() == 's':
        print("\nUtilizzo dei parametri per dati CON controllo termico.")
        # Parametri del modello stimato (originali)
        params = {
            'a1': 9.65e-05,      # Coefficiente loading
            'a2': 0.0511,        # Coefficiente unloading
            'a3': 0.0323,        # Coefficiente unload_light
            'h1': 0.0,           # Offset
            'm1': 2.0514,        # Esponente loading
            'm2': 1.0746,        # Esponente unloading
            'm3': 1.6057         # Esponente unload_light
        }
    else:
        print("\nUtilizzo dei parametri per dati SENZA controllo termico.")
        # Parametri del modello stimato (dalla tabella, colonna 'Mean')
        params = {
            'a1': 0.0001,
            'a2': 0.0247,
            'a3': 0.0057,
            'h1': 0.0,           # h0[1] dalla tabella
            'm1': 2.0420,
            'm2': 1.2138,
            'm3': 2.0921
        }

    # Inizializza l'analizzatore con i parametri scelti
    analyzer = ExperimentalDataAnalyzer(
        a1=params['a1'],
        a2=params['a2'],
        a3=params['a3'],
        h1=params['h1'],
        m1=params['m1'],
        m2=params['m2'],
        m3=params['m3']
    )
    
    # Esegue l'analisi completa
    results = analyzer.run_complete_analysis()

I dati sono stati ottenuti sotto controllo termico? (s/n):  s



Utilizzo dei parametri per dati CON controllo termico.
ANALIZZATORE DATI SPERIMENTALI - CONFRONTO CON MODELLO TEORICO


Inserisci il percorso completo del file dati sperimentali:
>  "C:\Users\geard\OneDrive\Download\Tesi\dati_ts\dataset_1\Prove BBI-68 VIDIT#Set3# 07.TXT"
Inserisci il percorso della cartella dove salvare i risultati:
>  "C:\Users\geard\OneDrive\Download\Tesi\Risultati"



File dati: C:\Users\geard\OneDrive\Download\Tesi\dati_ts\dataset_1\Prove BBI-68 VIDIT#Set3# 07.TXT
Cartella output: C:\Users\geard\OneDrive\Download\Tesi\Risultati

Dati caricati: 883 punti
Range temporale: 0.1 - 88.3 s
Range forza: 0.0001 - 10.0008 N
Range spostamento: 1.4 - 287.6 nm

Calcolo modello teorico...
Calcolo metriche...
Generazione grafici...
Salvataggio report...

ANALISI COMPLETATA

Metriche principali:
    Errore relativo medio: 9.81%
    Correlazione forma: 0.9451

File generati:
    Confronto temporale: C:\Users\geard\OneDrive\Download\Tesi\Risultati\analisi_Prove BBI-68 VIDIT#Set3# 07_curva_F-h.jpg
    Analisi errore: C:\Users\geard\OneDrive\Download\Tesi\Risultati\analisi_Prove BBI-68 VIDIT#Set3# 07_spostamento_tempo.jpg
    Report completo: C:\Users\geard\OneDrive\Download\Tesi\Risultati\analisi_Prove BBI-68 VIDIT#Set3# 07_report.txt
